In [ ]:
from typing import Callable

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.axes import Axes

import utils

plt.style.use('default')

In [ ]:
METRICS = ['LINE_COVERAGE', 'BRANCH_COVERAGE', 'RAW_MUT_SCORE', 'COV_MUT_SCORE']
CI = 95  # 95 % CI
ALPHA = 1 - CI / 100.0

RENDER = {
    'LINE_COVERAGE': 'Line-Coverage',
    'BRANCH_COVERAGE': 'Branch-Coverage',
    'RAW_MUT_SCORE': 'Roher Mutation-Score',
    'COV_MUT_SCORE': 'Coverage Mutation-Score',
    'LINE_COVERAGE_DIFF': '$\\Delta$ Line-Coverage',
    'BRANCH_COVERAGE_DIFF': '$\\Delta$ Branch-Coverage',
    'RAW_MUT_SCORE_DIFF': '$\\Delta$ Roher Mutation-Score',
    'COV_MUT_SCORE_DIFF': '$\\Delta$ Coverage Mutation-Score',
    'COVERAGE': 'Coverage',
    'MUTATION': 'Mutation',
}

df = utils.union_read_csv('data/Gson_iterations.csv', 'data/Lang_iterations.csv', 'data/JacksonCore_iterations.csv')
df = df[df['RUN_COMPLETED'] == True]
df['RAW_MUT_SCORE'] = df['KILLED_MUTANTS'] * 1.0 / df['MUTANTS']
df['COV_MUT_SCORE'] = df['KILLED_MUTANTS'] * 1.0 / df['COVERED_MUTANTS']

In [ ]:
print('D4J Bugs with no mutants:')
missing_mutants = df[df['MUTANTS'] == 0][['D4J_PROJECT', 'D4J_BUG_ID', 'CLASS_NAME']].drop_duplicates()
for m in missing_mutants.values:
    if len(df[(df['D4J_PROJECT'] == m[0]) & (df['D4J_BUG_ID'] == m[1])][
               ['D4J_PROJECT', 'D4J_BUG_ID', 'CLASS_NAME', 'MUTANTS']].dropna().drop_duplicates()) <= 1:
        print(m)

In [ ]:
def plot_metrics(
        axes: Axes,
        aggregated_per_metric: dict[str, pd.DataFrame],
        *,
        ylim: tuple[float, float] = (0, 1),
        colors: list[str] | None = None,
        format_value: Callable[[float], str] = lambda y: f'{y:.2f}',
        ylabel: str = 'Score',
):
    if colors is None:
        colors = [f'C{i}' for i in range(len(aggregated_per_metric.items()))]
    if len(colors) < len(aggregated_per_metric.items()):
        raise ValueError('Not enough colors to plot metrics')

    axes.axhline(y=0, linewidth=.5, color='k')
    for (metric_name, agg), color in zip(aggregated_per_metric.items(), colors):
        x_values = agg['ITERATION']
        y_values = agg['mean']
        axes.plot(
            x_values,
            y_values,
            marker='o',
            label=f'{RENDER[metric_name]} (Arithmetisches Mittel, {CI} % CI)',
            color=color,
        )
        axes.fill_between(
            x_values,
            y_values - agg['hw'],
            y_values + agg['hw'],
            alpha=0.3,
            color=color,
        )

        for (x, y) in zip(x_values, y_values):
            axes.annotate(format_value(y), xy=(x, y), textcoords='offset points', xytext=(0, 5), ha='center', va='bottom')

    axes.set_xlabel('Iteration')
    axes.set_xticks(range(0, 11))
    axes.set_ylim(ylim)
    axes.set_ylabel(ylabel)
    axes.grid(linestyle='--', linewidth=.5, alpha=0.5)
    axes.legend()

In [ ]:
for ty in ['COVERAGE', 'MUTATION']:
    subset = df[df['FEEDBACK_TYPE'] == ty]
    fig, axes = plt.subplots(1, 1, figsize=utils.A4_LANDSCAPE_INCHES, dpi=300)
    plt.title(f'Verlauf {RENDER[ty]}-Feedback')
    agg_metrics = {}
    for metric in METRICS:
        agg_metrics[metric] = utils.aggregate_for_metric(
            subset,
            ['FEEDBACK_TYPE', 'ITERATION'],
            lambda grouped: grouped[metric],
            ALPHA
        )

    plot_metrics(axes, agg_metrics)

    # axes.twinx()
    # axes.plot()

    # plt.show()
    fig.savefig(f'out/metric-development-{ty}.svg', bbox_inches='tight')

    # for metric in METRICS:
    #     agg_metrics[metric]['metric'] = metric
    out = pd.concat(agg_metrics, names=['METRIC']).reset_index(level=0).drop('FEEDBACK_TYPE', axis=1)
    out['mean'] = out['mean'].apply(lambda x: f'{x:.4f}')
    out['hw'] = out['hw'].apply(lambda x: f'{x:.4f}')
    out = out.pivot(index='ITERATION', columns=['METRIC'])
    out.columns = [f'{metric}_{stat}' for stat, metric in out.columns]
    out = out[[f'{m}_{s}' for m in METRICS for s in ['mean', 'hw', 'n']]]
    out.to_csv(f'data/out/metric-development-{ty}.csv')

In [ ]:
diffs = (
    df.dropna(subset=['FEEDBACK_TYPE'])
    .pivot(
        index=['RUN_UUID', 'ITERATION'],
        columns='FEEDBACK_TYPE',
        values=METRICS
    )
    .reset_index()
)

for m in METRICS:
    diffs[f'{m}_DIFF'] = diffs[(m, 'MUTATION')] - diffs[(m, 'COVERAGE')]

fig, axes = plt.subplots(1, 1, figsize=utils.A4_LANDSCAPE_INCHES, dpi=300)
plt.title(f'Verlauf gepaarte Differenz Mutation-Feedback $-$ Coverage-Feedback')
agg_metrics = {}
for metric in METRICS:
    agg_metrics[f'{metric}_DIFF'] = utils.aggregate_for_metric(
        diffs,
        ['ITERATION'],
        lambda grouped: grouped[f'{metric}_DIFF'],
        ALPHA
    )

plot_metrics(axes, agg_metrics, ylim=(-.06, .06), format_value=lambda v: f'{v:.3f}', ylabel='$\\Delta$ Score')
plt.show()
fig.savefig(f'out/metric-delta-development.svg', bbox_inches='tight')

out = pd.concat(agg_metrics, names=['METRIC']).reset_index(level=0)
out['mean'] = out['mean'].apply(lambda x: f'{x:.5f}')
out['hw'] = out['hw'].apply(lambda x: f'{x:.5f}')
out = out.pivot(index='ITERATION', columns=['METRIC'])
out.columns = [f'{metric}_{stat}' for stat, metric in out.columns]
out = out[[f'{m}_DIFF_{s}' for m in METRICS for s in ['mean', 'hw', 'n']]]
out.to_csv(f'data/out/metric-delta-development.csv')